# Portfolio Optimization for Trading

This notebook demonstrates how to use portfolio optimization to efficiently allocate capital across multiple assets.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from src.portfolio.portfolio_optimizer import PortfolioOptimizer, OptimizationStrategy
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and configure portfolio optimization settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Portfolio optimization configuration
portfolio_config = {
    'covariance_method': 'ledoit_wolf',
    'target_return': 0.15,
    'risk_aversion': 1.0,
    'max_turnover': 0.2
}

# Initialize optimizer
optimizer = PortfolioOptimizer(
    config=portfolio_config,
    risk_free_rate=0.0,
    transaction_costs=0.001
)

## Data Preparation

Fetch historical data for multiple assets.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Define assets
assets = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'ADAUSDT', 'DOTUSDT']

# Fetch historical data for each asset
data = {}
for symbol in assets:
    asset_data = await fetcher.fetch_historical_data(symbol=symbol)
    data[symbol] = asset_data['close']

# Create returns DataFrame
prices_df = pd.DataFrame(data)
returns_df = prices_df.pct_change().dropna()

print("Data Summary:")
print(f"Number of assets: {len(assets)}")
print(f"Date range: {returns_df.index[0]} to {returns_df.index[-1]}")
print("\nReturns Statistics:")
print(returns_df.describe())

## Portfolio Analysis

Analyze different portfolio optimization strategies.

In [ ]:
def analyze_strategy(returns: np.ndarray, strategy: OptimizationStrategy) -> dict:
    """Analyze portfolio optimization strategy."""
    metrics = optimizer.optimize_portfolio(
        returns=returns,
        strategy=strategy
    )
    
    return {
        'weights': metrics.weights,
        'expected_return': metrics.expected_return,
        'volatility': metrics.volatility,
        'sharpe_ratio': metrics.sharpe_ratio,
        'var': metrics.var,
        'es': metrics.es,
        'diversification_ratio': metrics.diversification_ratio,
        'concentration': metrics.concentration
    }

# Analyze different strategies
strategies = [
    OptimizationStrategy.MEAN_VARIANCE,
    OptimizationStrategy.MIN_VARIANCE,
    OptimizationStrategy.RISK_PARITY,
    OptimizationStrategy.HIERARCHICAL_RISK_PARITY
]

results = {}
for strategy in strategies:
    results[strategy] = analyze_strategy(returns_df.values, strategy)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot weights
ax = axes[0, 0]
weights_df = pd.DataFrame(
    {s.value: results[s]['weights'] for s in strategies},
    index=assets
)
weights_df.plot(kind='bar', ax=ax)
ax.set_title('Portfolio Weights')
ax.set_ylabel('Weight')
plt.xticks(rotation=45)

# Plot risk-return
ax = axes[0, 1]
for strategy in strategies:
    ax.scatter(
        results[strategy]['volatility'],
        results[strategy]['expected_return'],
        label=strategy.value
    )
ax.set_xlabel('Volatility')
ax.set_ylabel('Expected Return')
ax.set_title('Risk-Return Profile')
ax.legend()

# Plot risk metrics
ax = axes[1, 0]
risk_metrics = pd.DataFrame({
    s.value: [results[s]['var'], results[s]['es'], results[s]['volatility']]
    for s in strategies
}, index=['VaR', 'ES', 'Volatility'])
risk_metrics.plot(kind='bar', ax=ax)
ax.set_title('Risk Metrics')
plt.xticks(rotation=45)

# Plot diversification metrics
ax = axes[1, 1]
div_metrics = pd.DataFrame({
    s.value: [results[s]['diversification_ratio'],
              results[s]['concentration']]
    for s in strategies
}, index=['Diversification Ratio', 'Concentration'])
div_metrics.plot(kind='bar', ax=ax)
ax.set_title('Diversification Metrics')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nStrategy Comparison:")
for strategy in strategies:
    print(f"\n{strategy.value}:")
    print(f"Expected Return: {results[strategy]['expected_return']:.4f}")
    print(f"Volatility: {results[strategy]['volatility']:.4f}")
    print(f"Sharpe Ratio: {results[strategy]['sharpe_ratio']:.2f}")

## Portfolio Rebalancing

Analyze portfolio rebalancing over time.

In [ ]:
def analyze_rebalancing(returns_df: pd.DataFrame, window: int = 60):
    """Analyze portfolio rebalancing."""
    weights_history = []
    metrics_history = []
    
    for i in range(window, len(returns_df), 20):
        # Get historical data
        historical_returns = returns_df.iloc[i-window:i].values
        
        # Optimize portfolio
        metrics = optimizer.optimize_portfolio(
            returns=historical_returns,
            strategy=OptimizationStrategy.MEAN_VARIANCE
        )
        
        weights_history.append({
            'date': returns_df.index[i],
            **{asset: weight for asset, weight in
               zip(assets, metrics.weights)}
        })
        
        metrics_history.append({
            'date': returns_df.index[i],
            'expected_return': metrics.expected_return,
            'volatility': metrics.volatility,
            'sharpe_ratio': metrics.sharpe_ratio
        })
    
    return pd.DataFrame(weights_history), pd.DataFrame(metrics_history)

# Analyze rebalancing
weights_df, metrics_df = analyze_rebalancing(returns_df)

# Plot results
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot weights over time
ax = axes[0]
weights_df.set_index('date').plot(ax=ax, kind='area', stacked=True)
ax.set_title('Portfolio Weights Over Time')
ax.set_ylabel('Weight')

# Plot metrics over time
ax = axes[1]
metrics_df.set_index('date').plot(ax=ax)
ax.set_title('Portfolio Metrics Over Time')

plt.tight_layout()
plt.show()

# Calculate turnover
turnover = np.abs(
    weights_df.iloc[1:][assets].values -
    weights_df.iloc[:-1][assets].values
).sum(axis=1).mean()

print(f"\nAverage Turnover: {turnover:.4f}")

## Risk Analysis

Analyze portfolio risk characteristics.

In [ ]:
def analyze_risk(returns_df: pd.DataFrame, weights: np.ndarray):
    """Analyze portfolio risk."""
    # Calculate portfolio returns
    portfolio_returns = (returns_df * weights).sum(axis=1)
    
    # Calculate drawdown
    cumulative_returns = (1 + portfolio_returns).cumprod()
    rolling_max = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - rolling_max) / rolling_max
    
    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Returns distribution
    ax = axes[0, 0]
    sns.histplot(portfolio_returns, ax=ax)
    ax.set_title('Returns Distribution')
    
    # Cumulative returns
    ax = axes[0, 1]
    cumulative_returns.plot(ax=ax)
    ax.set_title('Cumulative Returns')
    
    # Drawdown
    ax = axes[1, 0]
    drawdown.plot(ax=ax)
    ax.set_title('Drawdown')
    
    # Rolling volatility
    ax = axes[1, 1]
    portfolio_returns.rolling(window=20).std().plot(ax=ax)
    ax.set_title('Rolling Volatility (20-day)')
    
    plt.tight_layout()
    plt.show()
    
    # Print risk statistics
    print("Risk Statistics:")
    print(f"Annualized Return: {portfolio_returns.mean()*252:.4f}")
    print(f"Annualized Volatility: {portfolio_returns.std()*np.sqrt(252):.4f}")
    print(f"Sharpe Ratio: {portfolio_returns.mean()/portfolio_returns.std()*np.sqrt(252):.2f}")
    print(f"Maximum Drawdown: {drawdown.min():.4f}")
    print(f"VaR (95%): {np.percentile(portfolio_returns, 5):.4f}")
    print(f"Expected Shortfall: {portfolio_returns[portfolio_returns < np.percentile(portfolio_returns, 5)].mean():.4f}")

# Analyze risk for mean-variance portfolio
mean_var_weights = results[OptimizationStrategy.MEAN_VARIANCE]['weights']
analyze_risk(returns_df, mean_var_weights)

## Black-Litterman Analysis

Incorporate views using Black-Litterman model.

In [ ]:
# Market capitalization data (example values)
market_caps = np.array([500, 200, 100, 50, 30])  # Billions USD

# Define views
views = {
    (0, 1): 0.05,  # BTC will outperform ETH by 5%
    (2, 3): 0.03   # BNB will outperform ADA by 3%
}

# Define view confidences
view_confidences = {
    (0, 1): 0.6,
    (2, 3): 0.7
}

# Optimize with Black-Litterman
bl_metrics = optimizer.optimize_portfolio(
    returns=returns_df.values,
    strategy=OptimizationStrategy.BLACK_LITTERMAN,
    market_caps=market_caps,
    views=views,
    view_confidences=view_confidences
)

# Compare with market portfolio
market_weights = market_caps / market_caps.sum()

# Plot comparison
plt.figure(figsize=(10, 6))
width = 0.35
x = np.arange(len(assets))

plt.bar(x - width/2, market_weights, width, label='Market Weights')
plt.bar(x + width/2, bl_metrics.weights, width, label='BL Weights')

plt.xlabel('Assets')
plt.ylabel('Weight')
plt.title('Market vs Black-Litterman Weights')
plt.xticks(x, assets, rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

# Print comparison
print("\nPortfolio Comparison:")
comparison = pd.DataFrame({
    'Market': market_weights,
    'Black-Litterman': bl_metrics.weights
}, index=assets)
print(comparison)

print("\nBlack-Litterman Metrics:")
print(f"Expected Return: {bl_metrics.expected_return:.4f}")
print(f"Volatility: {bl_metrics.volatility:.4f}")
print(f"Sharpe Ratio: {bl_metrics.sharpe_ratio:.2f}")